In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
work_dir = "/content/drive/MyDrive/NLP/Assignment 5"
data_dir= "/content/drive/MyDrive/NLP/ngram_model"
os.makedirs(work_dir, exist_ok=True)
os.chdir(work_dir)

print("Working directory:", os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content/drive/MyDrive/NLP/Assignment 5


In [ ]:
import pickle
import csv
from collections import Counter
import math
import pandas as pd

In [ ]:

def good_turing_smoothing(counts_dict, vocab_size, n):
    """
    Apply Good-Turing smoothing to one n-gram model.

    counts_dict: dict of {tuple: count}, e.g. {('દાન',): 3, ('આપો',): 4}
    vocab_size: size of vocabulary
    n: order of n-gram (1=unigram, 2=bigram, etc.)
    """
    # Total counts of seen n-grams
    N = sum(counts_dict.values())
    U = len(counts_dict) if n == 1 else None

    # Frequency of frequency (Nc table)
    freq_of_freq = Counter(counts_dict.values())

    # N1 = number of n-grams that occurred exactly once
    N1 = freq_of_freq.get(1, 0)

    # Probability for unseen n-grams
    if n == 1:
        unseen_count = vocab_size - U
        unseen_prob = (N1 / N) / unseen_count if unseen_count > 0 else 0.0
    else:
        unseen_count = (vocab_size ** n) - len(counts_dict)
        unseen_prob = (N1 / N) / unseen_count if unseen_count > 0 else 0.0

    # Adjust probabilities for seen n-grams
    probs = {}
    for ngram, c in counts_dict.items():
        Nc = freq_of_freq.get(c, 0)
        Nc1 = freq_of_freq.get(c + 1, 0)

        if Nc > 0 and Nc1 > 0:
            c_star = (c + 1) * (Nc1 / Nc)
        else:
            c_star = c  # fallback if Nc1 missing

        prob = max(c_star / N, 0.0)  # avoid -0.0
        probs[ngram] = prob

    # Normalize so total probability = 1
    total_seen_prob = sum(probs.values())
    total_unseen_prob = unseen_prob * unseen_count
    total_mass = total_seen_prob + total_unseen_prob

    if total_mass > 0:
        scale = 1.0 / total_mass
        probs = {ng: p * scale for ng, p in probs.items()}
        unseen_prob *= scale

    return probs, unseen_prob





In [ ]:
def smooth_single_model(input_pkl, output_pkl, n):
    # Load counts
    with open(input_pkl, "rb") as f:
        counts = pickle.load(f)

    vocab_size=get_vocab_size(counts, n)
    # Apply smoothing
    smoothed_probs, unseen_prob = good_turing_smoothing(counts, vocab_size, n)

    # Save results


    smoothed_model = {
        "probs": smoothed_probs,
        "unseen_prob": unseen_prob
    }
    with open(output_pkl, "wb") as f:
        pickle.dump(smoothed_model, f)

    print(f"[INFO] Smoothed {n}-gram model saved to {output_pkl}")
    print(f"[INFO] Unseen probability = {unseen_prob:.6e}, Seen ngrams = {len(counts)}")



In [ ]:
def get_vocab_size(ngram_counts, ngram_size):
    if ngram_size == 1:
        return len(ngram_counts)
    else:
        # unique words from last token of each ngram
        return len(set([ngram[-1] for ngram in ngram_counts.keys()]))


In [ ]:

for i in range(1, 5):
  print(f"started  working on {i} gram")

  smooth_single_model(f"{data_dir}/final_{i}gram_counts.pkl", f"{work_dir}/{i}gram_smoothed.pkl",i)

  print(f"good turing done on {i} gram")




# Helper Functions

In [ ]:

def get_probability(ngram, smoothed_model):
    """Return probability of ngram tuple using smoothed model."""
    return smoothed_model["probs"].get(ngram, smoothed_model["unseen_prob"])

def sentence_log_prob(sentence_tokens, smoothed_model, n):
    """Compute log probability of a sentence using n-gram model (natural log)."""
    log_prob = 0.0
    padded_tokens = ['<s>']*(n-1) + sentence_tokens + ['</s>']  # pad start/end

    for i in range(len(sentence_tokens) + 1):
        ngram = tuple(padded_tokens[i:i+n])
        prob = get_probability(ngram, smoothed_model)
        log_prob += math.log(prob) if prob > 0 else math.log(1e-12)  # avoid log(0)

    return log_prob

def sentence_perplexity(sentence_tokens, smoothed_model, n):
    """Compute perplexity of a sentence."""
    log_prob = sentence_log_prob(sentence_tokens, smoothed_model, n)
    N = len(sentence_tokens) + 1  # include </s>
    perplexity = math.exp(-log_prob / N)
    return perplexity

# ---------------- CSV Processing ----------------
def compute_perplexity_csv(input_csv, smoothed_model_file, n, output_csv):
    # Load smoothed model
    with open(smoothed_model_file, "rb") as f:
        smoothed_model = pickle.load(f)

    results = []
    with open(input_csv, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        for row in reader:
            if len(row) == 0:
                continue
            sentence = row[0].strip()
            tokens = sentence.split()
            log_prob = sentence_log_prob(tokens, smoothed_model, n)
            perp = sentence_perplexity(tokens, smoothed_model, n)
            results.append([sentence, log_prob, perp])

    # Save results to CSV
    with open(output_csv, "w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["sentence", "log_probability", "perplexity"])
        writer.writerows(results)

    print(f"Results saved to {output_csv}")




In [ ]:
input_csv = f"{data_dir}/val_sentences.csv"
for i in range(1,5):
    smoothed_model_file = f"{work_dir}/{i}gram_smoothed.pkl"
    output_csv = f"{work_dir}/{i}gram_sentence_perplexity.csv"

    compute_perplexity_csv(input_csv, smoothed_model_file, i, output_csv)


Results saved to /content/drive/MyDrive/NLP/Assignment 5/1gram_sentence_perplexity.csv
Results saved to /content/drive/MyDrive/NLP/Assignment 5/2gram_sentence_perplexity.csv
Results saved to /content/drive/MyDrive/NLP/Assignment 5/3gram_sentence_perplexity.csv
Results saved to /content/drive/MyDrive/NLP/Assignment 5/4gram_sentence_perplexity.csv


In [ ]:


def combine_ngram_csvs(unigram_csv, bigram_csv,trigram_csv,quadgram_csv):
    # Load all CSVs into dictionaries keyed by sentence
    def load_csv(file):
        data = {}
        with open(file, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                sentence = row["sentence"].strip()
                log_prob = float(row["log_probability"])
                perp = float(row["perplexity"])
                data[sentence] = (log_prob, perp)
        return data

    uni_data = load_csv(f"{work_dir}/{unigram_csv}")
    bi_data = load_csv(f"{work_dir}/{bigram_csv}")
    tri_data = load_csv(f"{work_dir}/{trigram_csv}")
    quad_data = load_csv(f"{work_dir}/{quadgram_csv}")

    # quad_data = load_csv(quadgram_csv)



    # Combine all sentences (union of all keys)
    all_sentences = set(uni_data.keys()) | set(bi_data.keys())

    # Print in desired format
    for sentence in all_sentences:
        print(f"Sentence: {sentence}")
        if sentence in uni_data:
            print(f"  Unigram  -> LogProb: {uni_data[sentence][0]:.4f}, Perplexity: {uni_data[sentence][1]:.4f}")
        if sentence in bi_data:
            print(f"  Bigram   -> LogProb: {bi_data[sentence][0]:.4f}, Perplexity: {bi_data[sentence][1]:.4f}")
        if sentence in tri_data:
            print(f"  Trigram   -> LogProb: {tri_data[sentence][0]:.4f}, Perplexity: {tri_data[sentence][1]:.4f}")
        if sentence in quad_data:
            print(f"  Quadgram   -> LogProb: {quad_data[sentence][0]:.4f}, Perplexity: {quad_data[sentence][1]:.4f}")


        print()  # blank line between sentences

# ---------------- Example Usage ----------------



In [ ]:
combine_ngram_csvs(
    "1gram_sentence_perplexity.csv",
    "2gram_sentence_perplexity.csv",
    "3gram_sentence_perplexity.csv".
    "4gram_sentence_perplexity.csv"
    )

Sentence: વસંત આજે અપુર્વને રાત્રે ઘરે આવવા કહે છે . .
  Unigram  -> LogProb: -87.5948, Perplexity: 2873.1491
  Bigram   -> LogProb: -138.0552, Perplexity: 282227.4270
  Trigram   -> LogProb: -195.0048, Perplexity: 50008848.0677

Sentence: proud of you .
  Unigram  -> LogProb: -32.8315, Perplexity: 710.7291
  Bigram   -> LogProb: -51.7679, Perplexity: 31369.0032
  Trigram   -> LogProb: -92.0294, Perplexity: 98530150.3825

Sentence: ઘરમાં શાંતિનું વાતાવરણ રહેશે .
  Unigram  -> LogProb: -50.6444, Perplexity: 4631.9243
  Bigram   -> LogProb: -71.0999, Perplexity: 140081.4314
  Trigram   -> LogProb: -114.7509, Perplexity: 202276549.7679

Sentence: Read Think and try to relate the truth during the vaidik age Vs 21st Century .
  Unigram  -> LogProb: -156.4128, Perplexity: 9904.5626
  Bigram   -> LogProb: -207.9898, Perplexity: 205806.7848
  Trigram   -> LogProb: -269.6999, Perplexity: 7761626.7963

Sentence: બધાજ વિષયો ઉપર વાચન કરી આપના પ્રતિભાવો અવશ્ય જણાવશો . હું રાહ જોઈશ્ . nLikeLike n જવ

In [ ]:
def load_counter_from_pkl(pkl_file):
    """
    Load a Counter from a pickle file.

    The .pkl file should contain a dictionary {ngram_tuple: count}.
    Returns a Counter object.
    """
    with open(pkl_file, "rb") as f:
        data = pickle.load(f)

    # Ensure keys are tuples and values are ints
    counter_data = Counter({tuple(k) if isinstance(k, (list,str)) else k: int(v) for k, v in data.items()})
    return counter_data

In [ ]:
# Cell 7: Task 3 - Good-Turing Frequency Tables

def good_turing_table(counter, top_k=100):
    # Build frequency-of-frequency
    freq_of_freq = Counter(counter.values())
    rows = []

    for c in sorted(freq_of_freq.keys())[:top_k]:
        Nc = freq_of_freq[c]
        Nc1 = freq_of_freq.get(c+1, 0)
        if Nc > 0:
            c_star = (c+1) * Nc1 / Nc
        else:
            c_star = c
        rows.append((c, Nc, c_star))

    df = pd.DataFrame(rows, columns=["c", "Nc", "c*"])
    return df


In [ ]:

# Generate tables
unigram_c=load_counter_from_pkl(f"{data_dir}/final_1gram_counts.pkl")
bigram_c=load_counter_from_pkl(f"{data_dir}/final_2gram_counts.pkl")
trigram_c=load_counter_from_pkl(f"{data_dir}/final_1gram_counts.pkl")
quadgram_c=load_counter_from_pkl(f"{data_dir}/final_2gram_counts.pkl")

uni_table  = good_turing_table(unigram_c)
bi_table   = good_turing_table(bigram_c)
tri_table  = good_turing_table(trigram_c)
quad_table   = good_turing_table(quadgram_c)

In [ ]:
print("Top Good-Turing frequencies for Unigrams:")
display(uni_table.head(100))

print("\nTop Good-Turing frequencies for Bigrams:")
display(bi_table.head(100))

print("Top Good-Turing frequencies for Trigrams:")
display(tri_table.head(100))

print("\nTop Good-Turing frequencies for Quadgrams:")
display(quad_table.head(100))

Top Good-Turing frequencies for Unigrams:


,c,Nc,c*
0,1,463245,0.397690
1,2,92114,1.431216
2,3,43945,2.587507
3,4,28427,3.266437
4,5,18571,4.882451
...,...,...,...
95,96,134,111.477612
96,97,154,75.090909
97,98,118,114.940678
98,99,137,87.591241



Top Good-Turing frequencies for Bigrams:


,c,Nc,c*
0,1,3587627,0.355247
1,2,637247,1.178816
2,3,250399,2.261031
3,4,141540,2.847322
4,5,80602,4.931788
...,...,...,...
95,96,281,72.491103
96,97,210,106.400000
97,98,228,85.973684
98,99,198,119.191919


Top Good-Turing frequencies for Trigrams:


,c,Nc,c*
0,1,463245,0.397690
1,2,92114,1.431216
2,3,43945,2.587507
3,4,28427,3.266437
4,5,18571,4.882451
...,...,...,...
95,96,134,111.477612
96,97,154,75.090909
97,98,118,114.940678
98,99,137,87.591241



Top Good-Turing frequencies for Quadgrams:


,c,Nc,c*
0,1,3587627,0.355247
1,2,637247,1.178816
2,3,250399,2.261031
3,4,141540,2.847322
4,5,80602,4.931788
...,...,...,...
95,96,281,72.491103
96,97,210,106.400000
97,98,228,85.973684
98,99,198,119.191919
